# 🏗️ Civil/Arch Statics · 💻 Digital Design · 🌌 Star Map · 🔬 Biotech · ∂ Calculus · 🧵 Systems

> **Chopped Griffiths format** — one concept → one equation → one number → one plot per §.x

| § | Topic | Boss equation |
|---|---|---|
| §1 | Statics ABET CA | ΣF=0, ΣM=0, shear-moment, seismic V=CsW |
| §2 | Moving-frame physics | Doppler, SR γ, Snell mirage, laser range |
| §3 | Digital: Verilog/C/Kmap/Power | CMOS P=αCV²f, Kmap SOP, timing slack |
| §4 | Star map 2025–2035 | RA/Dec → Alt/Az, planet ephemeris |
| §5 | Torch complex exp + Jalali | e^{iωt}, FFT, sech pulse, FNO |
| §6 | 7 Biotech investing Qs | Jalali lab framework: TAM, IP, runway |
| §7 | Related rates + log diff | dy/dt chain rule, ln-diff y=x^x |
| §8 | Threads + Cuckoo + GIC | Python threading, hash, interrupt ctrl |

In [ ]:
import numpy as np
import sympy as sp
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import matplotlib.gridspec as gridspec
from scipy.integrate import solve_ivp
from scipy.optimize import fsolve
import itertools, threading, time, hashlib, math, warnings, random
warnings.filterwarnings('ignore')
sp.init_printing(use_latex='mathjax')
from IPython.display import display, Math, Markdown
plt.rcParams.update({'font.size':10,'figure.dpi':100})
np.random.seed(2026)
print('Civil/Digital/Space/Biotech — all modules loaded')

---
## §1 🏗️ Statics ABET — California Architectural/Civil Engineering

**§1.1 Equilibrium:** $\sum F_x=0,\ \sum F_y=0,\ \sum M_O=0$

**§1.2 Truss — Method of Joints:** solve pin forces; $F>0$ tension, $F<0$ compression.

**§1.3 Beam — Shear & Moment:**
$$V(x) = V_0 - \int_0^x w(\xi)\,d\xi, \quad M(x) = M_0 + \int_0^x V(\xi)\,d\xi$$

**§1.4 California Seismic (ASCE 7):**
$V = C_s W$, $C_s = S_{DS}/(R/I_e)$, $S_{DS}$ = design spectral accel, $R$ = response mod.

**§1.5 Mohr's Circle:** principal stresses $\sigma_{1,2} = \bar\sigma \pm R$, $R=\sqrt{(\Delta\sigma/2)^2+\tau^2}$

In [ ]:
# §1 — Statics: Truss + Beam + Seismic + Mohr

# ── §1.2 Simple Pratt truss (4-panel, pin-jointed) ───────────────
# Nodes: 0(0,0) 1(3,0) 2(6,0) 3(9,0) 4(1.5,3) 5(4.5,3) 6(7.5,3)
nodes = np.array([[0,0],[3,0],[6,0],[9,0],[1.5,3],[4.5,3],[7.5,3]], dtype=float)
# Members: (i,j)
members = [(0,1),(1,2),(2,3),(0,4),(4,1),(1,5),(5,2),(2,6),(6,3),
           (4,5),(5,6)]
# Supports: node 0 pin (Rx,Ry), node 3 roller (Ry)
# Loads: node 1: -20kN, node 2: -20kN (downward)
loads = {1:[0,-20e3], 2:[0,-20e3]}

# Solve by stiffness method (direct)
n_nodes = len(nodes); n_dof = 2*n_nodes
n_mem   = len(members)
K_glob  = np.zeros((n_dof, n_dof))

def member_stiffness(n1, n2, EA=1e8):
    dx = nodes[n2,0]-nodes[n1,0]; dy = nodes[n2,1]-nodes[n1,1]
    L  = np.sqrt(dx**2+dy**2)
    c,s= dx/L, dy/L
    k  = EA/L * np.array([[ c*c, c*s,-c*c,-c*s],
                           [ c*s, s*s,-c*s,-s*s],
                           [-c*c,-c*s, c*c, c*s],
                           [-c*s,-s*s, c*s, s*s]])
    return k, L, c, s

for m,(i,j) in enumerate(members):  # loop: assemble stiffness
    k,L,c,s = member_stiffness(i,j)
    dofs = [2*i,2*i+1,2*j,2*j+1]
    for a in range(4):
        for b in range(4):
            K_glob[dofs[a],dofs[b]] += k[a,b]

# Apply BCs: node0 fixed (dof 0,1), node3 roller (dof 7)
F_glob = np.zeros(n_dof)
for n,(fx,fy) in loads.items():
    F_glob[2*n] += fx; F_glob[2*n+1] += fy

fixed_dofs = [0,1,7]  # pin at 0: x,y; roller at 3: y
free_dofs  = [d for d in range(n_dof) if d not in fixed_dofs]
K_ff = K_glob[np.ix_(free_dofs, free_dofs)]
F_f  = F_glob[free_dofs]
U_f  = np.linalg.solve(K_ff, F_f)

U = np.zeros(n_dof)
for i,d in enumerate(free_dofs): U[d] = U_f[i]

# Member forces
forces = {}
for (i,j) in members:   # loop: member forces
    k,L,c,s = member_stiffness(i,j)
    u_e = np.array([U[2*i],U[2*i+1],U[2*j],U[2*j+1]])
    F_e = k @ u_e
    forces[(i,j)] = -F_e[0]*c - F_e[1]*s  # axial (tension+)

# ── §1.3 Simply supported beam with UDL ──────────────────────────
L_beam = 10.0  # m
w_udl  = 15e3   # N/m
x_b    = np.linspace(0, L_beam, 500)
R_A    = w_udl*L_beam/2
V_beam = R_A - w_udl*x_b
M_beam = R_A*x_b - 0.5*w_udl*x_b**2
V_max  = R_A; M_max = w_udl*L_beam**2/8
print(f'§1.3 Beam: V_max={V_max/1e3:.1f} kN, M_max={M_max/1e3:.1f} kN·m')

# ── §1.4 California Seismic ───────────────────────────────────────
# Site class D, SDC D (Los Angeles typical)
S_DS = 1.0   # Design spectral accel (g) — LA zone
R    = 3.5   # Response mod (ordinary moment frame)
Ie   = 1.0   # Importance factor
W    = 5e6   # Seismic weight (N)
Cs   = S_DS / (R/Ie)
Cs   = min(max(Cs, 0.044*S_DS*Ie), 0.5*S_DS/(R/Ie))
V_seismic = Cs * W
print(f'§1.4 Seismic: Cs={Cs:.4f}, V={V_seismic/1e3:.1f} kN  (LA, SDC-D)')

# ── §1.5 Mohr's circle ───────────────────────────────────────────
sigma_x, sigma_y, tau_xy = 80e6, -40e6, 60e6  # Pa
sigma_avg = (sigma_x+sigma_y)/2
R_mohr    = np.sqrt(((sigma_x-sigma_y)/2)**2 + tau_xy**2)
sigma_1   = sigma_avg + R_mohr
sigma_2   = sigma_avg - R_mohr
theta_p   = 0.5*np.degrees(np.arctan2(2*tau_xy, sigma_x-sigma_y))
print(f'§1.5 Mohr: σ1={sigma_1/1e6:.1f} MPa, σ2={sigma_2/1e6:.1f} MPa, θp={theta_p:.1f}°')

# ── Plots ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15,10))
gs  = gridspec.GridSpec(2,3,fig)

# Truss
ax_tr = fig.add_subplot(gs[0,0])
force_vals = np.array(list(forces.values()))
f_max = max(abs(force_vals)) if len(force_vals) else 1
for (i,j),F in forces.items():        # loop: draw members
    col = 'blue' if F > 0 else 'red'
    lw  = 1 + 3*abs(F)/f_max
    ax_tr.plot([nodes[i,0],nodes[j,0]],[nodes[i,1],nodes[j,1]],
               color=col, lw=lw)
    mx = (nodes[i,0]+nodes[j,0])/2; my = (nodes[i,1]+nodes[j,1])/2
    ax_tr.text(mx, my+0.15, f'{F/1e3:.1f}', fontsize=6, ha='center')
ax_tr.scatter(nodes[:,0],nodes[:,1],s=50,color='k',zorder=5)
for n,(fx,fy) in loads.items():
    ax_tr.annotate('',xy=(nodes[n,0],nodes[n,1]),
                    xytext=(nodes[n,0],nodes[n,1]+1.5),
                    arrowprops=dict(arrowstyle='->',color='green',lw=2))
ax_tr.set_title('§1.2 Pratt Truss\nBlue=tension, Red=comp')
ax_tr.set_aspect('equal'); ax_tr.grid(True,alpha=0.3)

# Shear-Moment
ax_v = fig.add_subplot(gs[0,1])
ax_v.plot(x_b, V_beam/1e3, 'b-', lw=2, label='Shear V (kN)')
ax_v.fill_between(x_b, V_beam/1e3, alpha=0.2, color='blue')
ax_v.set_xlabel('x (m)'); ax_v.set_ylabel('V (kN)')
ax_v.set_title(f'§1.3 Shear (UDL w={w_udl/1e3}kN/m, L={L_beam}m)')
ax_v.grid(True,alpha=0.3); ax_v.legend()
ax_m = fig.add_subplot(gs[0,2])
ax_m.plot(x_b, M_beam/1e3, 'r-', lw=2, label='Moment M (kN·m)')
ax_m.fill_between(x_b, M_beam/1e3, alpha=0.2, color='red')
ax_m.axhline(M_max/1e3,ls='--',color='k',lw=1,label=f'M_max={M_max/1e3:.0f}kN·m')
ax_m.set_xlabel('x (m)'); ax_m.set_ylabel('M (kN·m)')
ax_m.set_title('§1.3 Bending Moment'); ax_m.grid(True,alpha=0.3); ax_m.legend(fontsize=8)

# Seismic story forces (triangular distribution)
n_stories = 8; story_h = 4.0; W_story = W/n_stories
h_arr  = np.arange(1,n_stories+1)*story_h
Fx_arr = V_seismic * h_arr / h_arr.sum()  # simplified triangular
ax_s = fig.add_subplot(gs[1,0])
ax_s.barh(h_arr, Fx_arr/1e3, height=story_h*0.7, color='orange', alpha=0.8)
ax_s.set_xlabel('Story force (kN)'); ax_s.set_ylabel('Height (m)')
ax_s.set_title(f'§1.4 Seismic story forces\nV={V_seismic/1e3:.0f}kN, Cs={Cs:.3f}')
ax_s.grid(True,alpha=0.3)

# Mohr's circle
ax_mo = fig.add_subplot(gs[1,1])
theta_c = np.linspace(0,2*np.pi,200)
ax_mo.plot(sigma_avg/1e6+R_mohr/1e6*np.cos(theta_c),
           R_mohr/1e6*np.sin(theta_c), 'k-', lw=2)
ax_mo.plot([sigma_x/1e6,sigma_y/1e6],[tau_xy/1e6,-tau_xy/1e6],'ro-',ms=8,lw=2)
ax_mo.axhline(0,color='k',lw=0.5); ax_mo.axvline(sigma_avg/1e6,color='gray',ls='--',lw=0.5)
ax_mo.plot(sigma_1/1e6,0,'b^',ms=10,label=f'σ1={sigma_1/1e6:.0f}MPa')
ax_mo.plot(sigma_2/1e6,0,'bv',ms=10,label=f'σ2={sigma_2/1e6:.0f}MPa')
ax_mo.set_xlabel('σ (MPa)'); ax_mo.set_ylabel('τ (MPa)')
ax_mo.set_title(f'§1.5 Mohr\'s Circle (θp={theta_p:.1f}°)'); ax_mo.legend(fontsize=8)
ax_mo.set_aspect('equal'); ax_mo.grid(True,alpha=0.3)

# ACI beam design (rectangular, singly reinforced)
ax_aci = fig.add_subplot(gs[1,2])
fc = 28e6; fy = 420e6; b_w = 0.3; d_eff = 0.55  # Pa, Pa, m, m
phi = 0.9; beta1 = 0.85
rho_arr = np.linspace(0.002, 0.02, 200)
a_arr   = rho_arr*fy*d_eff/(0.85*fc)
Mn_arr  = phi*rho_arr*fy*b_w*d_eff**2*(1 - a_arr/(2*d_eff))/1e3  # kN·m
rho_bal = 0.85*beta1*fc/fy * (600/(600+fy/1e6))
ax_aci.plot(rho_arr*100, Mn_arr, 'g-', lw=2)
ax_aci.axvline(rho_bal*100,ls='--',color='r',lw=1.5,label=f'ρ_bal={rho_bal*100:.2f}%')
ax_aci.set_xlabel('ρ (%)'); ax_aci.set_ylabel('φMn (kN·m)')
ax_aci.set_title('§1.6 ACI 318 Beam capacity φMn vs ρ')
ax_aci.legend(fontsize=8); ax_aci.grid(True,alpha=0.3)
plt.suptitle('🏗️ §1: Statics ABET — California Civil/Arch', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## §2 🚄 Moving-Frame Physics — Doppler · SR · Mirage · Directed Energy

**§2.1 Relativistic Doppler:**
$f_{obs} = f_s\sqrt{\frac{1+\beta}{1-\beta}}$ (source approaching), $\beta=v/c$

**§2.2 SR time dilation / length contraction:**
$\gamma = 1/\sqrt{1-\beta^2}$, $\Delta t' = \gamma\Delta t$, $L' = L/\gamma$

**§2.3 Atmospheric refraction / mirage:**
$n(h) \approx 1 + n_0 e^{-h/H}$, Snell: $n\sin\theta = \text{const}$,
ray bends toward denser air → inferior mirage when $dn/dh > 0$ near ground.

**§2.4 Directed energy from moving platform:**
Beam divergence $\theta_D = 1.22\lambda/D$, atmospheric turbulence $r_0$ (Fried param),
Strehl ratio $S \approx e^{-(D/r_0)^{5/3}}$, range $R$ → irradiance $I = P/(R\theta_D)^2$.

In [ ]:
# §2 — Moving-frame physics
c = 3e8   # m/s

# §2.1 Relativistic Doppler: train approaching/receding
v_train = np.linspace(0, 0.95*c, 500)
beta    = v_train/c
f_s     = 440.0  # Hz (A4 note)
f_app   = f_s * np.sqrt((1+beta)/(1-beta+1e-15))  # approaching
f_rec   = f_s * np.sqrt((1-beta)/(1+beta+1e-15))  # receding

# Non-relativistic classical Doppler (for low speeds)
v_sound = 343.0  # m/s
v_low   = np.linspace(0, 100, 200)  # m/s (train)
f_class_app = f_s * (v_sound/(v_sound - v_low))
f_class_rec = f_s * (v_sound/(v_sound + v_low))

# §2.2 SR: gamma factor
gamma_arr = 1/np.sqrt(1-beta**2+1e-30)
# Train GPS correction: v=7.9km/s satellite
v_sat = 7.9e3  # m/s
beta_sat = v_sat/c
gamma_sat = 1/np.sqrt(1-beta_sat**2)
dt_sr  = (gamma_sat-1)*86400*1e6  # microseconds/day time DILATION
# Gravitational blueshift (GPS): +45.9 μs/day; SR: -7.2 μs/day
print(f'§2.2 SR GPS time dilation: -{dt_sr:.2f} μs/day (SR, v=7.9km/s)')
print(f'     Net GPS correction: +{45.9-dt_sr:.1f} μs/day (gravity wins)')

# §2.3 Atmospheric refraction / mirage: ray tracing
# Exponential index profile n(h) = 1 + n0*exp(-h/H)
n0 = 3e-4; H = 8500  # scale height m
h_arr = np.linspace(0, 5000, 1000)  # 0 to 5 km
n_atm = 1 + n0*np.exp(-h_arr/H)
# Inferior mirage: temperature inversion near ground — inverted n profile
n_inv = 1 + n0*(1 + 2*np.exp(-h_arr/50))  # hot ground layer, n decreases upward

# Ray tracing: 2D, d/ds(n dr/ds) = grad(n)
def trace_ray(h0, theta0_deg, n_profile, h_grid, n_steps=2000):
    dh = h_grid[1]-h_grid[0]
    h, y = h0, 0.0
    theta = np.radians(theta0_deg)
    hs,ys = [h],[y]
    ds = 10.0  # step m
    for _ in range(n_steps):           # loop: ray trace
        idx = np.clip(int(h/dh),0,len(h_grid)-2)
        dn_dh = (n_profile[idx+1]-n_profile[idx])/dh
        n_h   = n_profile[idx]
        dtheta = -dn_dh/n_h * np.cos(theta) * ds
        theta += dtheta
        h  += np.sin(theta)*ds
        y  += np.cos(theta)*ds
        h   = max(h, 0.0)
        hs.append(h); ys.append(y)
        if h > 5000 or y > 50000: break
    return np.array(ys), np.array(hs)

# §2.4 Directed energy: irradiance vs range
lam_laser = 1064e-9  # Nd:YAG
D_aperture = np.array([0.1, 0.2, 0.3, 0.5])  # m
P_laser    = 100e3  # 100 kW
r_Fried    = 0.10   # Fried parameter 10 cm (moderate turbulence)
R_arr = np.logspace(2, 5, 300)  # 100 m to 100 km
irrad = {}
for D in D_aperture:           # loop: aperture sizes
    theta_D  = 1.22*lam_laser/D
    Strehl   = np.exp(-(D/r_Fried)**(5/3))
    I_ideal  = P_laser/(np.pi*(R_arr*theta_D/2)**2)
    irrad[D] = I_ideal * Strehl

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15,8))

# Doppler: acoustic (low v) and relativistic
axes[0][0].plot(v_low, f_class_app, 'b-', lw=2, label='Approach (classical)')
axes[0][0].plot(v_low, f_class_rec, 'r-', lw=2, label='Recede (classical)')
axes[0][0].axhline(f_s,ls='--',color='k',lw=1,label=f'f_source={f_s}Hz')
axes[0][0].set_xlabel('v_train (m/s)'); axes[0][0].set_ylabel('f_obs (Hz)')
axes[0][0].set_title('§2.1 Acoustic Doppler (train)')
axes[0][0].legend(fontsize=8); axes[0][0].grid(True,alpha=0.3)

axes[0][1].plot(beta*100, f_app/f_s,'b-',lw=2,label='Relativistic approach')
axes[0][1].plot(beta*100, f_rec/f_s,'r-',lw=2,label='Relativistic recede')
axes[0][1].set_xlabel('β = v/c (%)'); axes[0][1].set_ylabel('f_obs/f_s')
axes[0][1].set_title('§2.1 Relativistic Doppler')
axes[0][1].legend(fontsize=8); axes[0][1].grid(True,alpha=0.3)

# SR gamma
axes[0][2].semilogy(beta*100, gamma_arr,'g-',lw=2)
axes[0][2].set_xlabel('β = v/c (%)'); axes[0][2].set_ylabel('γ (Lorentz factor)')
axes[0][2].set_title('§2.2 SR Lorentz factor γ(β)')
axes[0][2].grid(True,alpha=0.3)
for bv in [0.5,0.9,0.99]:
    gv = 1/np.sqrt(1-bv**2)
    axes[0][2].annotate(f'β={bv}\nγ={gv:.2f}',(bv*100,gv),fontsize=7,
                         textcoords='offset points',xytext=(5,5))

# Mirage ray trace
ys_norm,hs_norm = trace_ray(100, 0.5, n_atm, h_arr)
ys_mirg,hs_mirg = trace_ray(1,  -0.3, n_inv, h_arr)
axes[1][0].plot(ys_norm/1e3, hs_norm, 'b-', lw=2, label='Normal atmosphere')
axes[1][0].plot(ys_mirg/1e3, hs_mirg, 'r-', lw=2, label='Mirage (inversion)')
axes[1][0].set_xlabel('Horizontal range (km)'); axes[1][0].set_ylabel('Height (m)')
axes[1][0].set_title('§2.3 Atmospheric ray tracing / mirage')
axes[1][0].legend(fontsize=8); axes[1][0].grid(True,alpha=0.3); axes[1][0].set_ylim(0,500)

# n(h) profiles
axes[1][1].plot((n_atm-1)*1e6, h_arr/1e3,'b-',lw=2,label='Normal n(h)')
axes[1][1].plot((n_inv-1)*1e6, h_arr/1e3,'r-',lw=2,label='Inversion (mirage)')
axes[1][1].set_xlabel('(n-1) × 10⁶'); axes[1][1].set_ylabel('h (km)')
axes[1][1].set_title('§2.3 Refractivity profiles'); axes[1][1].legend(fontsize=8)
axes[1][1].set_ylim(0,2); axes[1][1].grid(True,alpha=0.3)

# Directed energy irradiance
for D,I in irrad.items():          # loop: plot irradiance curves
    Strehl = np.exp(-(D/r_Fried)**(5/3))
    axes[1][2].loglog(R_arr/1e3, I/1e4, lw=2, label=f'D={D*100:.0f}cm S={Strehl:.2f}')
axes[1][2].axhline(1, ls='--', color='r', lw=1, label='1 W/cm² (damage)')
axes[1][2].set_xlabel('Range (km)'); axes[1][2].set_ylabel('Irradiance (W/cm²)')
axes[1][2].set_title(f'§2.4 Directed energy 100kW laser\nr0={r_Fried*100:.0f}cm')
axes[1][2].legend(fontsize=7); axes[1][2].grid(True,alpha=0.3)
plt.suptitle('🚄 §2: Moving-Frame Physics', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## §3 💻 Digital Design — Verilog · C · Makefile · Kmap · Timing · CMOS Power

**§3.1 CMOS Power:** $P_{dyn} = \alpha C V^2 f$, $P_{static} = V_{DD} I_{leak}$

**§3.2 Kmap SOP:** group 1s in $2^k$ cells → minimal sum-of-products

**§3.3 Setup/Hold timing slack:**
$t_{slack} = T_{clk} - t_{setup} - t_{prop} - t_{skew}$

**§3.4 Verilog snippet** (4-bit ripple adder) + **Makefile** + **C** model

In [ ]:
# §3 — Digital Design

# §3.1 CMOS Power analysis
V_DD_arr = np.linspace(0.5, 3.3, 100)
alpha    = 0.15   # activity factor
C_load   = 10e-15 # 10 fF
f_clk    = 1e9    # 1 GHz
I_leak   = 100e-9 # 100 nA/gate × ... (per gate normalized)
P_dyn    = alpha * C_load * V_DD_arr**2 * f_clk * 1e9  # nW per gate × 1e9 gates
P_stat   = V_DD_arr * I_leak * 1e9
P_total  = P_dyn + P_stat

# Technology scaling: Dennard scaling
nodes_nm = np.array([250,180,130,90,65,45,32,22,14,10,7,5,3])
P_norm   = (nodes_nm/250)**2  # quadratic scaling of dynamic power (approx)
perf_nm  = 250/nodes_nm       # relative performance

# §3.2 Kmap: 4-variable (A,B,C,D) — full 16-cell
# Example: F = Σm(0,2,5,7,8,10,13,15) — "gray code like"
minterms = {0,2,5,7,8,10,13,15}
truth    = [1 if i in minterms else 0 for i in range(16)]
# Build Kmap in Gray code order: AB\CD
gray2 = [0,1,3,2]  # 00,01,11,10
kmap  = np.zeros((4,4),dtype=int)
for row,ab in enumerate(gray2):   # loop: fill kmap
    for col,cd in enumerate(gray2):
        idx = (ab<<2)|cd
        kmap[row,col] = truth[idx]

# SOP by prime implicants (Petrick simplified for demo)
print('§3.2 Kmap F = Σm(0,2,5,7,8,10,13,15):')
print('     SOP = A\'C\' + AC + B\'D + BD  (cover by inspection)')

# §3.3 Timing: pipeline setup/hold
T_clk_arr = np.linspace(1, 20, 100)  # ns
t_prop    = 4.0   # ns combinational delay
t_setup   = 0.3   # ns flip-flop setup
t_hold    = 0.1   # ns
t_skew    = 0.2   # ns
t_slack   = T_clk_arr - t_prop - t_setup - t_skew
f_max     = 1/(t_prop+t_setup+t_skew)*1e9/1e6  # MHz
print(f'§3.3 Timing: f_max = {f_max:.0f} MHz  (t_prop={t_prop}ns, setup={t_setup}ns, skew={t_skew}ns)')

# Verilog source as text
verilog_4bit_adder = '''
// 4-bit ripple-carry adder (Verilog)
module adder4(input  [3:0] A, B,
              input        cin,
              output [3:0] S,
              output       cout);
  wire c1, c2, c3;
  full_adder FA0(.a(A[0]),.b(B[0]),.cin(cin), .s(S[0]),.cout(c1));
  full_adder FA1(.a(A[1]),.b(B[1]),.cin(c1),  .s(S[1]),.cout(c2));
  full_adder FA2(.a(A[2]),.b(B[2]),.cin(c2),  .s(S[2]),.cout(c3));
  full_adder FA3(.a(A[3]),.b(B[3]),.cin(c3),  .s(S[3]),.cout(cout));
endmodule

module full_adder(input a,b,cin, output s,cout);
  assign s    = a ^ b ^ cin;
  assign cout = (a&b)|(b&cin)|(a&cin);
endmodule
'''

makefile_src = '''
# Makefile for Verilog simulation with iverilog
SRC  = adder4.v
TB   = adder4_tb.v
SIM  = sim_adder4

all: $(SIM)

$(SIM): $(SRC) $(TB)
\t@iverilog -o $@ $^
\t@vvp $@

clean:
\t@rm -f $(SIM) *.vcd

.PHONY: all clean
'''

c_model = '''
// C model: 4-bit adder (golden reference for co-simulation)
#include <stdio.h>
#include <stdint.h>

uint8_t adder4(uint8_t A, uint8_t B, uint8_t cin) {
    return ((A & 0xF) + (B & 0xF) + (cin & 1)) & 0x1F;
}

int main(void) {
    for (int a=0; a<16; a++)
        for (int b=0; b<16; b++) {
            uint8_t r = adder4(a, b, 0);
            printf("A=%d B=%d -> S=%d cout=%d\\n", a, b, r&0xF, (r>>4)&1);
        }
    return 0;
}
'''
print('§3.4 Verilog 4-bit ripple adder (source displayed below)')
print(verilog_4bit_adder)

# §3.3 Simulate timing diagram: CLK, D, Q (D-FF)
t_sim  = np.linspace(0, 8, 10000)
T_clk_s= 2.0
CLK    = (np.sin(2*np.pi*t_sim/T_clk_s) > 0).astype(float)
D_sig  = np.array([0,0,1,1,0,1,1,0]*1250)[:10000].astype(float)
# Q captures D at rising edge (delayed by 0.05ns prop)
Q_sig  = np.zeros(10000)
for i in range(1,10000):       # loop: simulate FF
    if CLK[i]>CLK[i-1]: Q_sig[i:] = D_sig[i]; break
# simplified: Q = D delayed by half period
idx_prop = int(0.05/8*10000)
Q_sig    = np.roll(D_sig, idx_prop)

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15,8))

# CMOS power vs VDD
axes[0][0].plot(V_DD_arr, P_dyn, 'b-', lw=2, label='P_dyn')
axes[0][0].plot(V_DD_arr, P_stat,'r--',lw=2, label='P_static')
axes[0][0].plot(V_DD_arr, P_total,'k-',lw=2, label='P_total')
axes[0][0].axvline(1.2,ls=':',color='gray',label='1.2V (modern)')
axes[0][0].set_xlabel('VDD (V)'); axes[0][0].set_ylabel('Power (arb)')
axes[0][0].set_title('§3.1 CMOS Power vs VDD'); axes[0][0].legend(fontsize=8)
axes[0][0].grid(True,alpha=0.3)

# Technology scaling
axes[0][1].loglog(nodes_nm, P_norm,'ro-',ms=6,lw=2,label='Norm power')
axes[0][1].loglog(nodes_nm, perf_nm,'b^-',ms=6,lw=2,label='Perf (1/λ)')
axes[0][1].set_xlabel('Node (nm)'); axes[0][1].set_ylabel('Normalized')
axes[0][1].set_title('§3.1 Dennard scaling (power wall ~65nm)')
axes[0][1].legend(fontsize=8); axes[0][1].grid(True,alpha=0.3)
axes[0][1].invert_xaxis()

# Kmap
axes[0][2].imshow(kmap, cmap='Greens', vmin=0, vmax=1, aspect='auto')
for i in range(4):              # loop: annotate kmap
    for j in range(4):
        axes[0][2].text(j,i,str(kmap[i,j]),ha='center',va='center',fontsize=14,
                         fontweight='bold',color='white' if kmap[i,j] else 'black')
axes[0][2].set_xticks([0,1,2,3]); axes[0][2].set_xticklabels(['00','01','11','10'])
axes[0][2].set_yticks([0,1,2,3]); axes[0][2].set_yticklabels(['00','01','11','10'])
axes[0][2].set_xlabel('CD'); axes[0][2].set_ylabel('AB')
axes[0][2].set_title('§3.2 Kmap F(A,B,C,D)')

# Timing diagram
t_d   = np.linspace(0,8,1000)
CLK_d = (np.sin(2*np.pi*t_d/T_clk_s)>0).astype(float)
D_d   = np.array([0]*200+[1]*200+[0]*100+[1]*300+[0]*200)*1.0
D_d   = np.resize(D_d,1000)
axes[1][0].step(t_d, CLK_d+2.5,'k-',lw=2,where='post',label='CLK')
axes[1][0].step(t_d, D_d+1.2,  'b-',lw=2,where='post',label='D')
axes[1][0].step(t_d, np.roll(D_d,30)+0.0,'r-',lw=2,where='post',label='Q (after FF)')
axes[1][0].set_xlabel('Time (ns)'); axes[1][0].set_yticks([0.5,1.7,3.0])
axes[1][0].set_yticklabels(['Q','D','CLK'])
axes[1][0].set_title('§3.3 Timing diagram (D-FF)'); axes[1][0].grid(True,alpha=0.3)

# Timing slack vs T_clk
axes[1][1].plot(T_clk_arr, t_slack,'g-',lw=2)
axes[1][1].axhline(0,color='r',ls='--',lw=1.5,label='slack=0 (max freq)')
axes[1][1].fill_between(T_clk_arr,t_slack,0,where=(t_slack>0),alpha=0.2,color='green',label='Positive slack')
axes[1][1].fill_between(T_clk_arr,t_slack,0,where=(t_slack<0),alpha=0.2,color='red',label='Timing violation')
axes[1][1].set_xlabel('T_clk (ns)'); axes[1][1].set_ylabel('Slack (ns)')
axes[1][1].set_title(f'§3.3 Timing slack (f_max={f_max:.0f}MHz)')
axes[1][1].legend(fontsize=8); axes[1][1].grid(True,alpha=0.3)

# Verilog display
axes[1][2].text(0.02,0.98,verilog_4bit_adder.strip(),
                transform=axes[1][2].transAxes,
                fontsize=6,verticalalignment='top',fontfamily='monospace',
                bbox=dict(boxstyle='round',facecolor='#f0f0f0',alpha=0.9))
axes[1][2].axis('off'); axes[1][2].set_title('§3.4 Verilog: 4-bit adder')
plt.suptitle('💻 §3: Digital Design — Verilog/C/Kmap/Timing/Power', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print('Makefile:', makefile_src)
print('C model:', c_model)

---
## §4 🌌 Star Map 2025–2035 — Civil Engineering Astronomy

**§4.1 RA/Dec → Alt/Az** (local sidereal time LST):
$$\sin(\text{Alt}) = \sin\delta\sin\phi + \cos\delta\cos\phi\cos(H)$$
$$\tan(\text{Az}) = \frac{\sin H}{\cos H\sin\phi - \tan\delta\cos\phi}$$

**§4.2 Planet positions (Keplerian + JPL mean elements)** — 10-year ephemeris

**§4.3 Celestial surveying** — Polaris azimuth correction, elongation method

In [ ]:
# §4 — Star map 2025-2035
# Observer: Los Angeles, CA
lat_deg = 34.05; lon_deg = -118.25
lat = np.radians(lat_deg)

def jd_from_ymd(y,m,d,h=0):
    '''Julian date from year/month/day/hour.'''
    if m<=2: y-=1; m+=12
    A=int(y/100); B=2-A+int(A/4)
    return int(365.25*(y+4716))+int(30.6001*(m+1))+d+h/24+B-1524.5

def gmst_hours(jd):
    '''Greenwich Mean Sidereal Time in hours.'''
    T=(jd-2451545.0)/36525
    g=280.46061837+360.98564736629*(jd-2451545)+T**2*0.000387933-T**3/38710000
    return (g%360)/15

def radec_to_altaz(ra_h, dec_deg, jd, lat_deg, lon_deg):
    '''Convert RA/Dec to Alt/Az for given JD and observer.'''
    lst     = (gmst_hours(jd) + lon_deg/15) % 24
    H       = np.radians((lst - ra_h)*15)
    dec     = np.radians(dec_deg)
    lat     = np.radians(lat_deg)
    sin_alt = np.sin(dec)*np.sin(lat) + np.cos(dec)*np.cos(lat)*np.cos(H)
    alt     = np.degrees(np.arcsin(np.clip(sin_alt,-1,1)))
    az_num  = -np.cos(dec)*np.cos(lat)*np.sin(H)
    az_den  = np.sin(dec) - np.sin(lat)*sin_alt
    az      = np.degrees(np.arctan2(az_num, az_den)) % 360
    return alt, az

# Bright stars catalog (RA hours, Dec deg)
STARS = {
    'Sirius':     (6.753,  -16.72),
    'Canopus':    (6.400,  -52.70),
    'Arcturus':   (14.261,  19.18),
    'Vega':       (18.615,  38.78),
    'Capella':    (5.278,   45.99),
    'Rigel':      (5.242,   -8.20),
    'Procyon':    (7.655,    5.22),
    'Betelgeuse': (5.919,    7.41),
    'Altair':     (19.847,   8.87),
    'Aldebaran':  (4.599,   16.51),
    'Spica':      (13.420, -11.16),
    'Antares':    (16.490, -26.43),
    'Polaris':    (2.530,   89.26),
}

# Planet mean orbital elements (simplified, J2000 ecliptic)
PLANETS = {
    'Mercury': {'a':0.387,'e':0.205,'i':7.00, 'Omega':48.3,'omega':29.1,'L0':252.3,'n':4.092},
    'Venus':   {'a':0.723,'e':0.007,'i':3.39, 'Omega':76.7,'omega':54.9,'L0':181.9,'n':1.602},
    'Mars':    {'a':1.524,'e':0.093,'i':1.85, 'Omega':49.6,'omega':286.5,'L0':355.5,'n':0.524},
    'Jupiter': {'a':5.203,'e':0.049,'i':1.30, 'Omega':100.5,'omega':273.9,'L0':34.4,'n':0.0831},
    'Saturn':  {'a':9.537,'e':0.057,'i':2.49, 'Omega':113.7,'omega':339.4,'L0':50.1,'n':0.0335},
}

def planet_ecliptic_lon(p, jd):
    '''Approximate planet ecliptic longitude in degrees.'''
    T = (jd - 2451545.0)/365.25
    M = (p['L0'] + p['n']*T*365.25 - p['omega']) % 360
    M_rad = np.radians(M)
    # Equation of center (first-order)
    C = (2*p['e'] - p['e']**3/4)*np.sin(M_rad) + 5/4*p['e']**2*np.sin(2*M_rad)
    return (p['L0'] + p['n']*T*365.25 + np.degrees(C)) % 360

# Generate 10-year ephemeris
years = np.linspace(2025, 2035, 200)
jds   = np.array([jd_from_ymd(int(y), 1+int((y%1)*12), 1) for y in years])

planet_lons = {name: np.array([planet_ecliptic_lon(p,jd) for jd in jds])
               for name,p in PLANETS.items()}

# Star positions for Jan 1, 2026 midnight
jd_now = jd_from_ymd(2026,1,1,0)
star_altaz = {}
for star,(ra,dec) in STARS.items():  # loop: star positions
    # Simple radec_to_altaz without walrus operator
    lst   = (gmst_hours(jd_now) + lon_deg/15) % 24
    H     = np.radians((lst - ra)*15)
    dec_r = np.radians(dec)
    sin_alt_s = np.sin(dec_r)*np.sin(lat) + np.cos(dec_r)*np.cos(lat)*np.cos(H)
    alt_s = np.degrees(np.arcsin(np.clip(sin_alt_s,-1,1)))
    az_num= -np.cos(dec_r)*np.cos(lat)*np.sin(H)
    az_den= np.sin(dec_r)-np.sin(lat)*sin_alt_s
    az_s  = np.degrees(np.arctan2(az_num,az_den)) % 360
    star_altaz[star] = (alt_s, az_s)

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15,5))

# Star chart (polar Alt/Az)
ax_sc = plt.subplot(131, projection='polar')
for star,(alt,az) in star_altaz.items():   # loop: plot stars
    if alt > 0:
        r = 90-alt; th = np.radians(az)
        ax_sc.plot(th,r,'*',ms=10,color='gold')
        ax_sc.text(th,r+3,star,fontsize=6,ha='center')
ax_sc.set_rlim(0,90); ax_sc.set_theta_zero_location('N')
ax_sc.set_theta_direction(-1)
ax_sc.set_title('§4.1 Star chart LA\nJan 1 2026 midnight',pad=15)

# Planet longitudes 2025-2035
axes[1].set_title('§4.2 Planet ecliptic longitudes 2025–2035')
for name,lons in planet_lons.items():  # loop: planet tracks
    axes[1].plot(years, lons, lw=2, label=name)
axes[1].set_xlabel('Year'); axes[1].set_ylabel('Ecliptic longitude (°)')
axes[1].legend(fontsize=8); axes[1].grid(True,alpha=0.3)

# Polaris tracking for celestial surveying
t_hours = np.linspace(0,24,200)
jd_base = jd_from_ymd(2026,6,21,0)  # summer solstice
polaris_alt_track = []
polaris_az_track  = []
for h in t_hours:              # loop: hourly Polaris position
    jd_h  = jd_base + h/24
    lst_h = (gmst_hours(jd_h) + lon_deg/15) % 24
    H_pol = np.radians((lst_h - 2.530)*15)
    dec_pol = np.radians(89.26)
    sin_a = np.sin(dec_pol)*np.sin(lat)+np.cos(dec_pol)*np.cos(lat)*np.cos(H_pol)
    alt_p = np.degrees(np.arcsin(np.clip(sin_a,-1,1)))
    az_n  = -np.cos(dec_pol)*np.cos(lat)*np.sin(H_pol)
    az_d  = np.sin(dec_pol)-np.sin(lat)*sin_a
    az_p  = np.degrees(np.arctan2(az_n,az_d))%360
    polaris_alt_track.append(alt_p)
    polaris_az_track.append(az_p)
axes[2].plot(t_hours, polaris_az_track,'b-',lw=2,label='Azimuth')
axes[2].axhline(360,ls='--',color='k',lw=0.5)
axes[2].set_xlabel('Hour (UT)'); axes[2].set_ylabel('Polaris Azimuth (°)')
axes[2].set_title('§4.3 Polaris Az (LA, Jun 21 2026)\nfor celestial surveying')
axes[2].grid(True,alpha=0.3)
ax2b = axes[2].twinx()
ax2b.plot(t_hours,polaris_alt_track,'r--',lw=1.5,label='Altitude')
ax2b.set_ylabel('Altitude (°)',color='red')
plt.suptitle('🌌 §4: Star Map 2025–2035 — Civil Astronomy', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'§4.3 Polaris altitude LA: {np.mean(polaris_alt_track):.2f}° ≈ lat={lat_deg}° (confirming true north)')

---
## §5 🔬 Torch Complex Exponentials + Jalali Lab Biophotonics

**§5.1 Complex phasor:** $e^{i\omega t} = \cos\omega t + i\sin\omega t$

**§5.2 Sech pulse (Jalali ultrafast):**
$E(t) = A_0\,\text{sech}(t/T_0)\,e^{i(\omega_0 t + \phi_{NL}(t))}$

**§5.3 GVD dispersed pulse:** $\tilde{E}(\omega) = \tilde{E}_0(\omega)\,e^{i\beta_2\omega^2 z/2}$

**§5.4 FROG/SPIDER spectrogram** (time-frequency for pulse characterization)

In [ ]:
# §5 — Torch complex exponentials + Jalali biophotonics
dtype_c = torch.complex128

N   = 4096
T0  = 1e-12      # 1 ps pulse width
lam0= 1550e-9    # center wavelength
c   = 3e8
omega0 = 2*np.pi*c/lam0

dt = 5e-15  # 5 fs time step
t_arr  = torch.arange(-N//2, N//2, dtype=torch.float64)*dt
omega_arr = torch.fft.fftfreq(N, d=dt)*2*np.pi

# §5.1 Batched complex exponentials
freqs = torch.tensor([0.5e12, 1.0e12, 2.0e12, 5.0e12])  # THz
phasors = torch.exp(1j * 2*np.pi * freqs[:,None] * t_arr[None,:])  # (4, N)
print(f'§5.1 Phasor batch: {phasors.shape}, dtype={phasors.dtype}')
print(f'     |e^iωt| = {phasors.abs().mean():.6f} (should be 1.0)')

# §5.2 Sech pulse (unchirped)
E0_np  = 1.0/np.cosh(t_arr.numpy()/T0)
E_t    = torch.tensor(E0_np, dtype=torch.float64).to(dtype_c)

# §5.3 GVD dispersion (b2 = -21 ps²/km, z = 100m SMF)
b2  = -21e-27    # s²/m (SMF-28 at 1550nm)
z_vals = [0, 0.1, 0.5, 1.0, 5.0]  # km → m via *1e3
E_dispersed = {}
E_om = torch.fft.fft(E_t)
for z_km in z_vals:          # loop: dispersion lengths
    z_m  = z_km*1e3
    H    = torch.exp(1j * (b2/2 * omega_arr**2 * z_m).to(torch.float64))
    E_d  = torch.fft.ifft(E_om * H)
    E_dispersed[z_km] = E_d

# §5.4 Spectrogram (FROG-like): |∫E(t)g(t-τ)e^-iωt dt|²
# Gate = Gaussian window
def spectrogram_FROG(E, t, n_tau=64, n_om=128):
    '''Compute FROG-like spectrogram.'''
    T_gate = 2e-12  # 2 ps gate
    tau_arr = np.linspace(-5e-12, 5e-12, n_tau)
    om_arr  = np.linspace(-5e13,  5e13,  n_om)
    S = np.zeros((n_tau, n_om))
    E_np = E.numpy()
    t_np = t.numpy()
    for i,tau in enumerate(tau_arr):   # loop: delay gates
        gate = np.exp(-((t_np-tau)/T_gate)**2)
        E_g  = E_np * gate
        for j,om in enumerate(om_arr):
            integrand = E_g * np.exp(-1j*om*t_np)
            S[i,j] = abs(np.trapz(integrand, t_np))**2
    return S, tau_arr, om_arr

S_frog, tau_arr, om_arr = spectrogram_FROG(E_t, t_arr, n_tau=48, n_om=80)

# §5.5 Nonlinear phase (SPM): φ_NL = γ P(t) z
gamma_NL = 1.2e-3  # W^-1/m (SMF)
P_peak   = 1000    # W
z_NL     = 0.1     # m
phi_NL   = gamma_NL * P_peak * E0_np**2 * z_NL
E_SPM    = E_t * torch.exp(1j * torch.tensor(phi_NL, dtype=torch.float64))

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15,8))

# §5.1 Phasors
t_plot = t_arr.numpy()*1e12
for i,f in enumerate(freqs.numpy()):  # loop: phasor plots
    axes[0][0].plot(t_plot[:200],phasors[i,:200].real.numpy(),lw=1.5,
                    label=f'{f/1e12:.1f}THz')
axes[0][0].set_xlabel('t (ps)'); axes[0][0].set_ylabel('Re[e^iωt]')
axes[0][0].set_title('§5.1 Complex phasors (THz)'); axes[0][0].legend(fontsize=7)
axes[0][0].grid(True,alpha=0.3)

# §5.2 Pulse intensity
I_pulse = E_t.abs().numpy()**2
axes[0][1].plot(t_plot, I_pulse,'b-',lw=2,label='sech² pulse')
axes[0][1].set_xlabel('t (ps)'); axes[0][1].set_ylabel('|E|²')
axes[0][1].set_title(f'§5.2 Sech pulse T0={T0*1e12:.0f}ps')
axes[0][1].grid(True,alpha=0.3)

# §5.3 GVD broadening
for z_km,E_d in E_dispersed.items():  # loop: dispersed pulses
    I_d = E_d.abs().numpy()**2
    axes[0][2].plot(t_plot, I_d/I_d.max(), lw=2, label=f'z={z_km}km')
axes[0][2].set_xlabel('t (ps)'); axes[0][2].set_ylabel('|E|² (normalized)')
axes[0][2].set_title(f'§5.3 GVD broadening b2={b2*1e27:.0f}ps²/km')
axes[0][2].legend(fontsize=7); axes[0][2].grid(True,alpha=0.3)

# §5.4 FROG spectrogram
im_f = axes[1][0].imshow(S_frog.T, aspect='auto', origin='lower',
                           extent=[tau_arr[0]*1e12,tau_arr[-1]*1e12,
                                   om_arr[0]/1e12,om_arr[-1]/1e12],
                           cmap='hot')
axes[1][0].set_xlabel('Delay τ (ps)'); axes[1][0].set_ylabel('ω/2π (THz offset)')
axes[1][0].set_title('§5.4 FROG-like spectrogram')
plt.colorbar(im_f,ax=axes[1][0])

# §5.5 SPM spectrum
E_SPM_om = torch.fft.fftshift(torch.fft.fft(E_SPM)).abs().numpy()**2
om_np    = torch.fft.fftshift(omega_arr).numpy()/(2*np.pi*1e12)
E_lin_om = torch.fft.fftshift(torch.fft.fft(E_t)).abs().numpy()**2
axes[1][1].plot(om_np, E_lin_om/E_lin_om.max(),'b-',lw=1.5,label='Input spectrum')
axes[1][1].plot(om_np, E_SPM_om/E_SPM_om.max(),'r-',lw=1.5,label='After SPM')
axes[1][1].set_xlim(-5,5); axes[1][1].set_xlabel('Δf (THz)')
axes[1][1].set_ylabel('Spectral power (norm)'); axes[1][1].legend(fontsize=8)
axes[1][1].set_title(f'§5.5 SPM spectral broadening\nγ={gamma_NL*1e3:.1f}/W/km, z={z_NL}m')
axes[1][1].grid(True,alpha=0.3)

# Jalali art: Bessel × dispersed sech (signature visual)
r_art = np.linspace(0,1,256); phi_art=np.linspace(0,2*np.pi,256)
R_art,PHI_art=np.meshgrid(r_art,phi_art)
from scipy.special import jv as jv_art
art = (jv_art(3,12*R_art)*np.cos(3*PHI_art) +
       jv_art(0, 8*R_art)*np.cos(5*PHI_art))*I_pulse[N//2-128:N//2+128].mean()
im_a=axes[1][2].imshow(art,cmap='twilight_shifted',origin='lower')
axes[1][2].set_title('§5 Jalali Art: Bessel×sech dispersion\n(Jalali Photonics Group signature)')
axes[1][2].axis('off'); plt.colorbar(im_a,ax=axes[1][2])
plt.suptitle('🔬 §5: Torch Complex Exp + Jalali Biophotonics', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## §6 💰 7 Questions for Investing in Biotech (Jalali Lab Framework)

Before writing a check, answer all 7:

| # | Question | Jalali-Lab Signal |
|---|---|---|
| **Q1** | **What is the unmet clinical need?** | Ultrafast imaging catches disease states invisible to slow cameras |
| **Q2** | **What is the TAM ($)?** | Diagnostic imaging: $50B; liquid biopsy: $8B; AI diagnostics: $20B |
| **Q3** | **What is the core IP moat?** | STEAM/sCMOS patents, compressed sensing algorithms, FNO models |
| **Q4** | **What is the FDA pathway?** | 510(k) De Novo vs PMA — time to revenue |
| **Q5** | **What does the runway look like?** | Burn rate < 12 mo runway = red flag; >24 mo = green |
| **Q6** | **Who are the co-investors / KOLs?** | Jalali group → UCLA Med collaborations = clinical validation |
| **Q7** | **What is the exit multiple?** | Acqui-hire vs strategic (GE/Siemens) vs IPO — 5–10× target |

In [ ]:
# §6 — 7 Biotech investing questions (quantitative models)

# Q2: TAM sizing (bottom-up)
TAM = {
    'Diagnostic Imaging': 50e9,
    'Liquid Biopsy':       8e9,
    'AI Diagnostics':     20e9,
    'Ultrafast Microscopy': 3e9,
    'Biophotonics':        5e9,
}

# Q4: FDA pathway timeline simulation
pathways = {
    '510(k) Exempt':   {'mean':6,  'std':2,  'color':'green'},
    '510(k) Standard': {'mean':12, 'std':4,  'color':'blue'},
    'De Novo':         {'mean':24, 'std':6,  'color':'orange'},
    'PMA':             {'mean':48, 'std':12, 'color':'red'},
}

# Q5: Runway model (burn rate Monte Carlo)
np.random.seed(42)
n_sims     = 5000
burn_rates = np.random.lognormal(np.log(500e3), 0.4, n_sims)   # $/month
cash       = np.random.lognormal(np.log(10e6), 0.5, n_sims)    # $
runway_mo  = cash / burn_rates   # months

# Q7: Exit multiple distribution (comparable biophotonics exits)
exit_multiples = np.array([
    3.2, 5.8, 7.1, 2.4, 12.0, 4.5, 8.3, 6.0, 15.0, 3.8,
    9.2, 4.1, 22.0, 6.7, 5.5, 11.0, 3.0, 7.8, 4.9, 18.0
])

# Scoring matrix: weight each Q 1-10
companies = {
    'JalaliCo (hypothetical)': [9,8,9,7,8,9,8],
    'Average Biotech A':       [6,5,5,6,5,5,6],
    'Undifferentiated B':      [4,7,3,5,3,4,4],
}
weights = [0.20, 0.15, 0.20, 0.10, 0.15, 0.10, 0.10]
Q_labels = ['Unmet Need','TAM','IP Moat','FDA Path','Runway','KOLs','Exit Multiple']

scores = {}
for co,s in companies.items():   # loop: compute weighted scores
    scores[co] = sum(w*v for w,v in zip(weights,s))

print('§6 Biotech Investment Scorecard:')
for co,sc in scores.items():
    print(f'  {co:35s}: {sc:.2f}/10')

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(15,8))

# TAM bubble chart
names = list(TAM.keys()); vals = list(TAM.values())
colors_tam = plt.cm.Set2(np.linspace(0,1,len(names)))
axes[0][0].bar(range(len(names)), [v/1e9 for v in vals], color=colors_tam)
axes[0][0].set_xticks(range(len(names))); axes[0][0].set_xticklabels(names,rotation=30,ha='right',fontsize=7)
axes[0][0].set_ylabel('TAM ($B)'); axes[0][0].set_title('Q2: TAM by segment')
axes[0][0].grid(True,alpha=0.3,axis='y')

# FDA pathway times
for i,(path,info) in enumerate(pathways.items()):  # loop: FDA timelines
    x = np.random.normal(info['mean'], info['std'], 1000)
    x = x[x>0]
    axes[0][1].violinplot(x, positions=[i], showmedians=True)
axes[0][1].set_xticks(range(len(pathways)))
axes[0][1].set_xticklabels(list(pathways.keys()),fontsize=8)
axes[0][1].set_ylabel('Time to clearance (months)')
axes[0][1].set_title('Q4: FDA Pathway Timelines'); axes[0][1].grid(True,alpha=0.3,axis='y')

# Runway distribution
axes[0][2].hist(runway_mo, bins=60, color='steelblue', edgecolor='k', lw=0.3)
axes[0][2].axvline(12,color='red',ls='--',lw=2,label='12mo (danger)')
axes[0][2].axvline(24,color='green',ls='--',lw=2,label='24mo (healthy)')
p_danger = (runway_mo<12).mean()
axes[0][2].set_xlabel('Runway (months)'); axes[0][2].set_ylabel('Count')
axes[0][2].set_title(f'Q5: Runway dist. P(<12mo)={p_danger:.1%}')
axes[0][2].legend(fontsize=8); axes[0][2].grid(True,alpha=0.3)

# Radar chart for scoring
angles = np.linspace(0, 2*np.pi, len(Q_labels), endpoint=False)
angles = np.concatenate([angles,[angles[0]]])
ax_r = plt.subplot(2,3,4, projection='polar')
colors_r = ['green','blue','red']
for (co,s),col in zip(companies.items(),colors_r):  # loop: radar
    vals_r = s + [s[0]]
    ax_r.plot(angles, vals_r, '-o', color=col, lw=2, ms=4, label=co[:15])
    ax_r.fill(angles, vals_r, alpha=0.1, color=col)
ax_r.set_xticks(angles[:-1])
ax_r.set_xticklabels(Q_labels,fontsize=7)
ax_r.set_ylim(0,10); ax_r.set_title('Q1-Q7 Scoring radar',pad=15)
ax_r.legend(fontsize=7,loc='upper right',bbox_to_anchor=(1.4,1.1))

# Exit multiple distribution
axes[1][1].hist(exit_multiples, bins=10, color='gold', edgecolor='k', lw=0.5)
axes[1][1].axvline(exit_multiples.mean(),color='r',ls='--',lw=2,label=f'Mean={exit_multiples.mean():.1f}x')
axes[1][1].axvline(5,color='b',ls=':',lw=2,label='5x target')
axes[1][1].set_xlabel('Exit multiple'); axes[1][1].set_ylabel('Count')
axes[1][1].set_title('Q7: Biophotonics exit multiples (n=20 comps)')
axes[1][1].legend(fontsize=8); axes[1][1].grid(True,alpha=0.3)

# Investment decision matrix
decision_data = np.zeros((3,7))
for i,(co,s) in enumerate(companies.items()):
    decision_data[i] = s
im_dm = axes[1][2].imshow(decision_data, cmap='RdYlGn', vmin=0, vmax=10, aspect='auto')
axes[1][2].set_xticks(range(7)); axes[1][2].set_xticklabels(Q_labels,rotation=45,ha='right',fontsize=7)
axes[1][2].set_yticks(range(3)); axes[1][2].set_yticklabels(list(companies.keys()),fontsize=7)
for i in range(3):
    for j in range(7):
        axes[1][2].text(j,i,f'{decision_data[i,j]:.0f}',ha='center',va='center',fontsize=9,fontweight='bold')
plt.colorbar(im_dm,ax=axes[1][2])
axes[1][2].set_title('§6 Biotech Q1-Q7 decision matrix')
plt.suptitle('💰 §6: 7 Biotech Investing Questions + Jalali Lab', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## §7 ∂ Related Rates · Chain Rule · Logarithmic Differentiation

**§7.1 Related rates — ladder problem:**
$x^2+y^2=L^2 \Rightarrow x\dot x+y\dot y=0 \Rightarrow \dot y = -x\dot x/y$

**§7.2 Chain rule:** $\frac{d}{dt}[f(g(t))] = f'(g)\cdot g'(t)$

**§7.3 Logarithmic differentiation:** $y=x^x \Rightarrow \ln y = x\ln x \Rightarrow y'/y = \ln x+1$

**§7.4 Related rates — water tank (cone):**
$V=\frac{1}{3}\pi r^2 h$, $r/h=R/H$ const $\Rightarrow \dot h = \frac{\dot V}{\pi r^2}$

In [ ]:
# §7 — Calculus: Related Rates, Chain Rule, Log Diff

x_s, y_s, t_s, h_s, r_s, L_s = sp.symbols('x y t h r L', positive=True)

# §7.1 Ladder problem (SymPy)
L_val = 10  # m ladder
eq_ladder = x_s**2 + y_s**2 - L_val**2
# dy/dt when x=6, dx/dt = 0.5 m/s
x_val = 6; y_val = np.sqrt(L_val**2 - x_val**2)
dxdt = 0.5
dydt = -x_val*dxdt/y_val
print(f'§7.1 Ladder: x={x_val}m, y={y_val:.3f}m, dx/dt={dxdt}, dy/dt={dydt:.4f} m/s')

# Parametric ladder fall
t_fall = np.linspace(0, y_val/abs(dydt), 200)
x_lad  = np.sqrt(np.maximum(0, L_val**2 - (y_val + dydt*t_fall)**2))
y_lad  = np.maximum(0, y_val + dydt*t_fall)
dydt_t = -x_lad*dxdt/(y_lad+1e-10)

# §7.2 Chain rule: SymPy symbolic
f_outer = sp.sin
g_inner = x_s**2 + 1
h_comp  = f_outer(g_inner)
dh_dx   = sp.diff(h_comp, x_s)
print('§7.2 Chain rule: d/dx[sin(x²+1)] =', sp.latex(dh_dx))
display(Math(r'\frac{d}{dx}[\sin(x^2+1)] = ' + sp.latex(dh_dx)))

# Chain rule gallery
funcs = [
    (sp.exp(x_s**2),         r'e^{x^2}'),
    (sp.ln(sp.sin(x_s)),     r'\ln(\sin x)'),
    (sp.sqrt(x_s**3+1),      r'\sqrt{x^3+1}'),
    (sp.cos(sp.exp(-x_s)),   r'\cos(e^{-x})'),
    (sp.atan(x_s**2+x_s),    r'\arctan(x^2+x)'),
]
print('\n§7.2 Chain rule gallery:')
for f,label in funcs:        # loop: chain rule
    df = sp.diff(f,x_s)
    print(f'  d/dx[{label}] = {sp.latex(sp.simplify(df))}')

# §7.3 Log differentiation
y_logdiff = x_s**x_s
# ln(y) = x*ln(x), d/dx → y'/y = ln(x)+1
dy_logdiff = y_logdiff*(sp.ln(x_s)+1)
display(Math(r'\frac{d}{dx}[x^x] = x^x(\ln x + 1) = ' + sp.latex(sp.simplify(sp.diff(y_logdiff,x_s)))))

# More log diff
log_funcs = [
    (x_s**sp.sin(x_s),           r'x^{\sin x}'),
    ((2*x_s+1)**sp.sqrt(x_s),    r'(2x+1)^{\sqrt{x}}'),
    (x_s**(x_s**x_s),             r'x^{x^x}'),
]
print('\n§7.3 Logarithmic differentiation:')
for f,label in log_funcs:    # loop: log diff
    df = sp.diff(f,x_s)
    print(f'  d/dx[{label}] = {sp.latex(sp.simplify(df)[:80])}...')

# §7.4 Cone water tank
R_tank = 2.0; H_tank = 4.0  # m
dVdt   = 0.5  # m³/s fill rate
h_fill = np.linspace(0.01, H_tank, 300)
r_fill = R_tank*h_fill/H_tank
V_fill = np.pi/3*r_fill**2*h_fill
dVdh   = np.pi*r_fill**2
dhdt   = dVdt/dVdh
print(f'\n§7.4 Cone tank: dh/dt at h={H_tank/2:.1f}m = {dVdt/(np.pi*(R_tank/2)**2):.4f} m/s')

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(15,8))

# §7.1 Ladder
for i,t in enumerate(np.linspace(0,t_fall[-1]*0.9,5)):  # loop: ladder snapshots
    xi = np.interp(t,t_fall,x_lad); yi = np.interp(t,t_fall,y_lad)
    axes[0][0].plot([0,xi],[yi,yi],'k-',lw=0.5,alpha=0.3)
    axes[0][0].plot([xi,xi],[0,yi],'k-',lw=0.5,alpha=0.3)
    axes[0][0].plot([0,xi],[yi,0],lw=2,alpha=0.6,label=f't={t:.1f}s')
axes[0][0].set_xlim(0,11); axes[0][0].set_ylim(0,11)
axes[0][0].set_xlabel('x (m)'); axes[0][0].set_ylabel('y (m)')
axes[0][0].set_title('§7.1 Falling ladder'); axes[0][0].legend(fontsize=7)
axes[0][0].set_aspect('equal'); axes[0][0].grid(True,alpha=0.3)

# dy/dt vs time
axes[0][1].plot(t_fall, dydt_t,'r-',lw=2)
axes[0][1].set_xlabel('t (s)'); axes[0][1].set_ylabel('dy/dt (m/s)')
axes[0][1].set_title('§7.1 dy/dt: ladder falls faster as y→0')
axes[0][1].grid(True,alpha=0.3)

# §7.2 Chain rule: f and f' on same plot
x_plot = np.linspace(-2,2,300)
g_np   = x_plot**2+1
h_np   = np.sin(g_np)
dh_np  = np.cos(g_np)*2*x_plot
axes[0][2].plot(x_plot,h_np,'b-',lw=2,label='sin(x²+1)')
axes[0][2].plot(x_plot,dh_np,'r--',lw=2,label="d/dx = 2x·cos(x²+1)")
axes[0][2].set_xlabel('x'); axes[0][2].set_title('§7.2 Chain rule: sin(x²+1)')
axes[0][2].legend(fontsize=8); axes[0][2].grid(True,alpha=0.3)

# §7.3 Log diff: x^x and derivative
x_ld = np.linspace(0.1,3,300)
y_xx   = x_ld**x_ld
dy_xx  = x_ld**x_ld*(np.log(x_ld)+1)
axes[1][0].plot(x_ld,y_xx, 'b-',lw=2,label='y = x^x')
axes[1][0].plot(x_ld,dy_xx,'r--',lw=2,label="y' = x^x(ln x + 1)")
axes[1][0].set_xlabel('x'); axes[1][0].set_ylim(-1,10)
axes[1][0].set_title('§7.3 Log diff: y = x^x')
axes[1][0].legend(); axes[1][0].grid(True,alpha=0.3)

# §7.4 Cone tank
axes[1][1].plot(h_fill, dhdt,'g-',lw=2)
axes[1][1].set_xlabel('h (m)'); axes[1][1].set_ylabel('dh/dt (m/s)')
axes[1][1].set_title(f'§7.4 Cone tank dh/dt (dV/dt={dVdt}m³/s)')
axes[1][1].grid(True,alpha=0.3)
ax74b=axes[1][1].twinx()
ax74b.plot(h_fill,V_fill,'b--',lw=1.5)
ax74b.set_ylabel('V (m³)',color='blue')

# §7.2 Implicit differentiation: circle x²+y²=r²
th = np.linspace(0,2*np.pi,200)
x_c=3*np.cos(th); y_c=3*np.sin(th)
axes[1][2].plot(x_c,y_c,'b-',lw=2,label='x²+y²=9')
# Tangent at (2, √5)
x0=2; y0=np.sqrt(5)
slope=-x0/y0
x_tan=np.linspace(x0-1.5,x0+1.5,50)
axes[1][2].plot(x_tan,y0+slope*(x_tan-x0),'r--',lw=2,label=f'Tangent slope={slope:.2f}')
axes[1][2].plot(x0,y0,'ko',ms=8)
axes[1][2].set_aspect('equal'); axes[1][2].grid(True,alpha=0.3)
axes[1][2].set_title('§7.1 Implicit diff: tangent to circle')
axes[1][2].legend(fontsize=8)
plt.suptitle('∂ §7: Related Rates · Chain Rule · Log Diff', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## §8 🧵 Threads · 🐦 Cuckoo Hash · ⚡ GIC · 🏗️ Statics Openings

**§8.1 Python threading:** race condition → Lock → Semaphore → producer-consumer

**§8.2 Cuckoo hashing:** two hash tables $h_1,h_2$; worst-case $O(1)$ lookup.
If collision: evict occupant, place in alternate table. Max load ~50%.

**§8.3 GIC (ARM Generic Interrupt Controller):**
Distributor → CPU Interface; priority (8-bit), preemption, SPI/PPI/SGI

**§8.4 Statics openings** (like chess openings for structural analysis):
- **Roller-Pin** beam: statically determinate, 3 unknowns = 3 eqs
- **Fixed-Fixed** beam: statically indeterminate, compatibility eqs required
- **Three-hinge arch**: statically determinate with extra $\sum M_{hinge}=0$

In [ ]:
# §8 — Threads, Cuckoo Hash, GIC, Statics Openings
import threading, time, queue, random, hashlib

# §8.1 Threading: race condition demo → Lock fix
counter_unsafe = 0
counter_safe   = 0
lock_c = threading.Lock()

def increment_unsafe(n):
    global counter_unsafe
    for _ in range(n):            # loop: unsafe increment
        tmp = counter_unsafe
        counter_unsafe = tmp + 1

def increment_safe(n):
    global counter_safe
    for _ in range(n):            # loop: safe increment
        with lock_c:
            counter_safe += 1

N_threads = 4; N_increments = 10000
threads_u = [threading.Thread(target=increment_unsafe, args=(N_increments,)) for _ in range(N_threads)]
threads_s = [threading.Thread(target=increment_safe,   args=(N_increments,)) for _ in range(N_threads)]
for t in threads_u: t.start()
for t in threads_u: t.join()
for t in threads_s: t.start()
for t in threads_s: t.join()
print(f'§8.1 Race condition (unsafe): {counter_unsafe} (expected {N_threads*N_increments})')
print(f'§8.1 With Lock (safe):        {counter_safe} (expected {N_threads*N_increments})')

# Producer-consumer with queue
results_queue = []
prod_times = []; cons_times = []
def producer(q, n):
    for i in range(n):            # loop: produce items
        t0=time.perf_counter()
        q.put(i)
        prod_times.append(time.perf_counter()-t0)

def consumer(q, n, out):
    for _ in range(n):            # loop: consume items
        t0=time.perf_counter()
        item = q.get()
        out.append(item)
        cons_times.append(time.perf_counter()-t0)

q_pc = queue.Queue(maxsize=10)
N_items = 200
out_items = []
t_prod = threading.Thread(target=producer, args=(q_pc, N_items))
t_cons = threading.Thread(target=consumer, args=(q_pc, N_items, out_items))
t_prod.start(); t_cons.start()
t_prod.join();  t_cons.join()
print(f'§8.1 Producer-consumer: {len(out_items)} items transferred')

# §8.2 Cuckoo Hash
class CuckooHash:
    def __init__(self, capacity=64):
        self.cap = capacity
        self.T1  = [None]*capacity
        self.T2  = [None]*capacity
        self.max_kicks = 32

    def _h1(self, k): return hash(k) % self.cap
    def _h2(self, k): return (hash(k)//self.cap + 7) % self.cap

    def insert(self, k, v):
        kicks = 0
        cur_k, cur_v = k, v
        while kicks < self.max_kicks:    # loop: cuckoo eviction
            i1 = self._h1(cur_k)
            if self.T1[i1] is None:
                self.T1[i1] = (cur_k, cur_v); return True
            cur_k, cur_v, self.T1[i1] = *self.T1[i1], (cur_k, cur_v)
            i2 = self._h2(cur_k)
            if self.T2[i2] is None:
                self.T2[i2] = (cur_k, cur_v); return True
            cur_k, cur_v, self.T2[i2] = *self.T2[i2], (cur_k, cur_v)
            kicks += 2
        return False  # need rehash

    def lookup(self, k):
        if (e:=self.T1[self._h1(k)]) and e[0]==k: return e[1]
        if (e:=self.T2[self._h2(k)]) and e[0]==k: return e[1]
        return None

# Benchmark: cuckoo vs Python dict
cht = CuckooHash(capacity=512)
keys_v = [f'key_{i}' for i in range(200)]
vals_v = list(range(200))
n_inserted = sum(1 for k,v in zip(keys_v,vals_v) if cht.insert(k,v))
n_found    = sum(1 for k,v in zip(keys_v,vals_v) if cht.lookup(k)==v)
print(f'§8.2 Cuckoo hash: {n_inserted}/200 inserted, {n_found}/200 verified')

# Load factor analysis
load_results = []
for load in np.linspace(0.1,0.9,9):     # loop: load factors
    cap = 1024; n_to_ins = int(cap*load)
    ch = CuckooHash(capacity=cap)
    n_ok = sum(1 for i in range(n_to_ins) if ch.insert(f'k{i}',i))
    load_results.append((load, n_ok/max(n_to_ins,1)))

loads_r, success_r = zip(*load_results)

# §8.3 GIC registers (ARM GICv2 simplified)
gic_regs = {
    'GICD_CTLR':   {'addr':0xE000_0000, 'desc':'Distributor control'},
    'GICD_ISENABLER0': {'addr':0xE000_0100,'desc':'Interrupt Set-Enable'},
    'GICD_IPRIORITYR0':{'addr':0xE000_0400,'desc':'Interrupt Priority'},
    'GICC_CTLR':   {'addr':0xE000_1000,'desc':'CPU Interface control'},
    'GICC_PMR':    {'addr':0xE000_1004,'desc':'Priority Mask'},
    'GICC_IAR':    {'addr':0xE000_100C,'desc':'Interrupt Acknowledge'},
    'GICC_EOIR':   {'addr':0xE000_1010,'desc':'End of Interrupt'},
}
print('\n§8.3 GIC Register Map (ARM GICv2):')
for reg,(info) in gic_regs.items():
    print(f'  {reg:25s} 0x{info["addr"]:08X}  {info["desc"]}')

# IRQ priority simulation: 8 interrupts with priorities
irqs = {'UART_RX':0x20,'Timer_IRQ':0x40,'GPIO_INT':0x60,
        'DMA_Done':0x30,'SPI_TX':0x80,'I2C_ERR':0x10,'USB_SOF':0x70,'NMI':0x00}
sorted_irqs = sorted(irqs.items(), key=lambda x: x[1])
print('\n§8.3 IRQ dispatch order (lower priority value = higher priority):')
for irq,prio in sorted_irqs:    # loop: dispatch
    print(f'  {irq:15s} priority=0x{prio:02X}')

# §8.4 Statics openings: fixed-fixed beam (compatibility)
# Fixed-fixed beam with central load P at x=a
E_beam=200e9; I_beam=1e-4; L_b=6.0; P_b=50e3; a_b=L_b/2
# Reactions (by superposition)
R_A_ff = P_b*(L_b-a_b)**2*(L_b+2*a_b)/L_b**3
R_B_ff = P_b*a_b**2*(L_b+2*(L_b-a_b))/L_b**3
M_A_ff = P_b*a_b*(L_b-a_b)**2/L_b**2
M_B_ff = P_b*a_b**2*(L_b-a_b)/L_b**2
print(f'\n§8.4 Fixed-Fixed beam (L={L_b}m, P={P_b/1e3}kN at mid):')
print(f'  R_A={R_A_ff/1e3:.2f}kN, R_B={R_B_ff/1e3:.2f}kN')
print(f'  M_A={M_A_ff/1e3:.2f}kN·m, M_B={M_B_ff/1e3:.2f}kN·m')

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(15,8))

# Thread timing
axes[0][0].hist(np.array(prod_times)*1e6, bins=30, alpha=0.6, label='Produce', color='blue')
axes[0][0].hist(np.array(cons_times)*1e6, bins=30, alpha=0.6, label='Consume', color='red')
axes[0][0].set_xlabel('Time (μs)'); axes[0][0].set_ylabel('Count')
axes[0][0].set_title('§8.1 Producer-Consumer latency'); axes[0][0].legend()
axes[0][0].grid(True,alpha=0.3)

# Cuckoo load factor
axes[0][1].plot([l*100 for l in loads_r], [s*100 for s in success_r], 'g-o', ms=8, lw=2)
axes[0][1].axvline(50, ls='--', color='r', lw=2, label='50% load (theoretical max)')
axes[0][1].set_xlabel('Load factor (%)'); axes[0][1].set_ylabel('Insert success (%)')
axes[0][1].set_title('§8.2 Cuckoo hash success vs load'); axes[0][1].legend(fontsize=8)
axes[0][1].grid(True,alpha=0.3)

# GIC priority waterfall
irq_names = [i for i,p in sorted_irqs]
irq_prios = [p for i,p in sorted_irqs]
axes[0][2].barh(irq_names, irq_prios, color=plt.cm.RdYlGn_r(np.array(irq_prios)/256))
axes[0][2].set_xlabel('Priority value (0=highest)')
axes[0][2].set_title('§8.3 GIC IRQ priority order'); axes[0][2].grid(True,alpha=0.3,axis='x')

# Fixed-Fixed beam moment diagram
x_ff = np.linspace(0, L_b, 400)
M_ff = np.where(x_ff<=a_b,
                -M_A_ff + R_A_ff*x_ff,
                -M_A_ff + R_A_ff*x_ff - P_b*(x_ff-a_b))
axes[1][0].plot(x_ff, M_ff/1e3,'r-',lw=2)
axes[1][0].fill_between(x_ff, M_ff/1e3, alpha=0.2, color='red')
axes[1][0].axhline(0,color='k',lw=0.5)
axes[1][0].set_xlabel('x (m)'); axes[1][0].set_ylabel('M (kN·m)')
axes[1][0].set_title('§8.4 Fixed-Fixed beam moment diagram'); axes[1][0].grid(True,alpha=0.3)

# Statics openings: support comparison
opening_data = {
    'Simply\nSupported': {'rxns':3,'eqs':3,'det':'Determinate'},
    'Cantilever':        {'rxns':3,'eqs':3,'det':'Determinate'},
    'Fixed-Fixed\nBeam': {'rxns':6,'eqs':3,'det':'Indeterminate (3°)'},
    'Propped\nCantilever':{'rxns':4,'eqs':3,'det':'Indeterminate (1°)'},
    'Three-Hinge\nArch': {'rxns':4,'eqs':4,'det':'Determinate'},
}
names_o = list(opening_data.keys())
rxns_o  = [d['rxns'] for d in opening_data.values()]
eqs_o   = [d['eqs']  for d in opening_data.values()]
colors_o= ['green' if r==e else 'red' for r,e in zip(rxns_o,eqs_o)]
axes[1][1].bar(range(len(names_o)), rxns_o, color=colors_o, alpha=0.7, label='Reactions')
axes[1][1].axhline(3,ls='--',color='k',lw=2,label='Equations of equilibrium (3)')
axes[1][1].set_xticks(range(len(names_o))); axes[1][1].set_xticklabels(names_o,fontsize=7)
axes[1][1].set_ylabel('Count'); axes[1][1].set_title('§8.4 Statics Openings: Det. vs Indet.')
axes[1][1].legend(fontsize=8); axes[1][1].grid(True,alpha=0.3,axis='y')

# Cuckoo hash visualization (T1, T2 occupancy)
cap_v = 64
ch_v  = CuckooHash(capacity=cap_v)
for i in range(30): ch_v.insert(f'word_{i}',i)
T1_occ = np.array([1 if ch_v.T1[i] else 0 for i in range(cap_v)])
T2_occ = np.array([1 if ch_v.T2[i] else 0 for i in range(cap_v)])
axes[1][2].bar(range(cap_v), T1_occ, alpha=0.7, color='blue', label='T1')
axes[1][2].bar(range(cap_v), T2_occ, alpha=0.5, color='red',  label='T2', bottom=T1_occ)
axes[1][2].set_xlabel('Bucket index'); axes[1][2].set_ylabel('Occupied')
axes[1][2].set_title(f'§8.2 Cuckoo hash (30 keys, cap={cap_v})')
axes[1][2].legend(fontsize=8); axes[1][2].grid(True,alpha=0.3,axis='y')
plt.suptitle('🧵 §8: Threads · Cuckoo · GIC · Statics Openings', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print('\nAll sections complete.')